# Agents

LLMs cannot reliably answer questions about current events from their training data alone. Tools such as web search let them retrieve up-to-date information.

## LangChain

LangChain provides interfaces for connecting language models to prompts and tools. This notebook uses LangChain to demonstrate tool calling.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI


load_dotenv()

llm = ChatOpenAI(model="gpt-4o", temperature=0)

response = llm.invoke("What is the date today?")
display(Markdown(response.content))


I'm sorry, but I can't provide real-time information. Please check your device or calendar for the current date.

## Providing today’s date in the prompt

We supply today’s date using Python’s `date.today()` and ask the model to answer a current-weather question to the best of its ability. This lets us observe its response with the date provided, before giving it access to web search.


In [2]:
from datetime import date
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful assistant. "
    f"Today's date is {date.today()}. "
    "Answer as accurately as you can."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{user_input}"),
])

chain = prompt | llm

question = (
    "What is the weather in Atlanta right now? "
    "Answer to the best of your ability using today's date "
    "and your existing knowledge. Explain any uncertainty."
)

response = chain.invoke({"user_input": question})
display(Markdown(response.content))


I'm unable to provide real-time weather updates as my training only includes data up to October 2023, and I don't have access to live data feeds. However, I can suggest that you check a reliable weather website or app for the most current weather information in Atlanta on November 20, 2026. Websites like the National Weather Service, Weather.com, or apps like AccuWeather can provide up-to-date forecasts and conditions.

## Adding a web-search tool

Now we give the model access to web search so it can retrieve current information before answering the same question. The model requests a search, our code runs the tool, and the results are passed back to the model to produce its response.


In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

from tools.search import web_search

messages = [
    SystemMessage(content=(
        f"Today's date is {date.today()}. "
        "Use web search to find current weather information. "
        "Include source links and explain any uncertainty."
    )),
    HumanMessage(content=question),
]

llm_with_tools = llm.bind_tools([web_search])
response = llm_with_tools.invoke(messages)

if response.tool_calls:
    messages.append(response)
    for tool_call in response.tool_calls:
        messages.append(web_search.invoke(tool_call))
    response = llm.invoke(messages)

display(Markdown(response.content))


I found some information about the weather in Atlanta for November 20, 2026. According to [World-Weather.info](https://world-weather.info/forecast/usa/atlanta_1/20-november/), the forecast includes details on temperature, wind, atmospheric pressure, humidity, and precipitation. However, this information is based on long-term forecasts and historical data, which can introduce some uncertainty.

Additionally, [AccuWeather](https://www.accuweather.com/en/us/atlanta/30303/november-weather/348181) suggests that the typical weather for November 20 in Atlanta includes a high temperature near 64°F, with a 77% chance of a dry day. This is based on historical averages from 1970 to 2026.

These sources provide a general idea of what the weather might be like, but real-time conditions can vary due to the inherent unpredictability of weather patterns. For the most accurate and up-to-date information, checking a local weather service or app would be advisable.

### How tool calling works

The model chooses whether to request a search and supplies the search arguments. Python executes the requested tool calls and returns their results to the model, which writes the final answer.

This example allows one round of tool calls. Web search provides retrieved information; the final answer still depends on the quality and freshness of the sources.
